# 03 · Detection Tuning

## Neden burası?
Baseline (`02_baseline`) sonuçları **jaccard ≈ node_recall²** olduğunu gösterdi
(FP ≈ 0 → linking hatasız; kenar kaybı tamamen tespit edilemeyen node'lardan).

| dataset | recall | recall² | jaccard |
|---|---|---|---|
| 44b6_0113de3b | 1.000 | 1.00 | 0.94 |
| 44b6_0b24845f | 0.333 | 0.11 | 0.18 |
| 6bba_05b6850b | 0.920 | 0.846 | 0.846 |
| 6bba_05db0fb1 | 0.881 | 0.777 | 0.75 |

→ **Recall'da +%1 ≈ jaccard'da +%2.** Bütün kazanç detection'da.

## İkinci problem: yoğunluk 11× oynak
`6.002 → 66.152` node/dataset (60 → 661 /kare). Beklenen ~213/kare.
Fazla-tahmin cezası: `jaccard × (1 − 0.1·(T_pred−T_true)/T_true)`.

## Hipotez (kök sebep)
`maximum_filter(size=...)` **tam pencere**, yarıçap değil. `(3,11,11)` = ±1 voxel Z, ±5 voxel XY.
Ama çekirdek yarıçapı ≈ **(3, 12, 12) voxel** (~5 µm) → pencere çekirdekten küçük →
**tek çekirdekte birden çok tepe**. `sigma=(1,2,2)` de (XY'de 0.8 µm) bunları birleştiremiyor.

## Yöntem
Tracking YOK. GT'li karelerden birkaçını **RAM'e yükle**, sonra parametre gridini
saniyeler içinde tara. Ölçüt: **recall** + **yoğunluk (node/kare)**.

## 0 · Kurulum

In [ ]:
import sys, subprocess, os, time, itertools
def _scan(base):
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if not d.endswith((".zarr",".geff")) and d!="competitions"]
        if root.count(os.sep) > 9: dirs[:]=[]; continue
        yield root, files
def ensure_zarr():
    try:
        import zarr; return zarr
    except ImportError: pass
    for root, files in _scan("/kaggle/input"):
        if os.path.basename(root)=="zarr" and "__init__.py" in files:
            p=os.path.dirname(root); sys.path.insert(0,p)
            try:
                import zarr; print("zarr <- sys.path:",p); return zarr
            except ImportError: sys.path.pop(0)
    subprocess.run([sys.executable,"-m","pip","install","-q","zarr"],check=False)
    import zarr; return zarr
zarr=ensure_zarr(); print("zarr:",zarr.__version__)

import numpy as np, pandas as pd
from pathlib import Path
from scipy import ndimage as ndi
from scipy.spatial.distance import cdist
from skimage.filters import threshold_otsu
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

SCALE=(1.625,0.40625,0.40625); S=np.array(SCALE,dtype=np.float32)
MATCH_UM=7.0
TARGET_DENS=213      # EDA: tahmini gercek cekirdek/kare
print("hazir")

In [ ]:
INPUT=Path("/kaggle/input")
def find_root():
    st=[(INPUT,0)]
    while st:
        b,d=st.pop()
        try:
            if (b/"train").is_dir() and (b/"test").is_dir(): return b
        except Exception: pass
        if d<4:
            for c in sorted(b.iterdir()):
                if c.is_dir() and not c.name.endswith((".zarr",".geff")): st.append((c,d+1))
ROOT=find_root(); TRAIN=ROOT/"train"; TEST=ROOT/"test"
test_names=sorted(p.stem for p in TEST.glob("*.zarr"))

def open_image(zp):
    n=zarr.open(str(zp),mode="r"); a=dict(n.attrs)
    ms=a.get("multiscales") or (a.get("ome") or {}).get("multiscales")
    if ms: return n[ms[0]["datasets"][0]["path"]]
    return n["0"] if "0" in list(n.keys()) else n

def load_geff(gp):
    g=zarr.open(str(gp),mode="r"); nodes=g["nodes"]
    ids=np.asarray(nodes["ids"]); props={}
    for pn in list(nodes["props"].keys()):
        try: props[pn]=np.asarray(nodes["props"][pn]["values"])
        except Exception: pass
    d={"id":ids}
    for k in ("t","z","y","x"):
        if k in props: d[k]=props[k]
    return pd.DataFrame(d)
print("test:",test_names)

## 1 · GT'li kareleri RAM'e yükle
Her dataset'ten GT'si en yoğun **K kare**. Bir kez okunur, tüm konfigürasyonlar bunu kullanır.

In [ ]:
K=4     # dataset basina kare (RAM: ~17MB/kare)
FRAMES=[]   # (dataset, t, volume, gt_points)
t0=time.time()
for nm in test_names:
    gdf=load_geff(TRAIN/(nm+".geff"))
    arr=open_image(TEST/(nm+".zarr"))
    cnt=gdf.groupby("t").size().sort_values(ascending=False)
    picks=[int(t) for t in cnt.index[:K]]        # GT'si en yogun kareler
    for t in picks:
        v=np.asarray(arr[t]).astype(np.float32)
        gp=gdf[gdf.t==t][["z","y","x"]].values.astype(np.float32)
        FRAMES.append((nm,t,v,gp))
print(f"{len(FRAMES)} kare yuklendi ({time.time()-t0:.0f}s)")
print("RAM ~%.0f MB"%(sum(f[2].nbytes for f in FRAMES)/1e6))
for nm,t,v,gp in FRAMES: print(f"  {nm} t={t:3d} GT={len(gp):4d}")

## 2 · Detection fonksiyonu (parametrik)

In [ ]:
def detect(v, sigma, foot, thr_mode):
    sm=ndi.gaussian_filter(v, sigma=sigma)
    if thr_mode=="otsu":        thr=threshold_otsu(sm)
    elif thr_mode.startswith("otsu*"): thr=threshold_otsu(sm)*float(thr_mode.split("*")[1])
    elif thr_mode.startswith("p"):     thr=np.percentile(sm, float(thr_mode[1:]))
    else: raise ValueError(thr_mode)
    mx=ndi.maximum_filter(sm, size=foot)
    peaks=(sm==mx)&(sm>thr)
    lbl,n=ndi.label(peaks)
    if n==0: return np.zeros((0,3),np.float32)
    return np.asarray(ndi.center_of_mass(sm,lbl,np.arange(1,n+1)),dtype=np.float32)

def recall_density(cents, gtpts):
    if len(cents)==0: return 0.0, 0
    D=cdist(gtpts*S, cents*S)          # um
    return float((D.min(1)<=MATCH_UM).mean()), len(cents)
print("ok")

## 3 · Parametre taraması
`sigma` çekirdeği yumuşatır (tek tepe), `foot` tepe ayrımının minimum mesafesini belirler.
Çekirdek yarıçapı ≈ (3,12,12) voxel; komşu aralığı ≈ 10–12 µm ≈ (7,25,25) voxel.

In [ ]:
SIGMAS=[(1,2,2), (1.5,4,4), (1.8,7,7)]
FOOTS =[(3,11,11), (5,15,15), (5,21,21), (7,25,25)]
THRS  =["otsu", "otsu*0.8", "p97"]

rows=[]; t0=time.time()
for sg,ft,th in itertools.product(SIGMAS,FOOTS,THRS):
    r=[]; d=[]; per_ds={}
    for nm,t,v,gp in FRAMES:
        c=detect(v,sg,ft,th); rec,den=recall_density(c,gp)
        r.append(rec); d.append(den); per_ds.setdefault(nm,[]).append(rec)
    rec=float(np.mean(r)); den=float(np.mean(d))
    pen=1.0 if den<=TARGET_DENS else max(0.0,1-0.1*(den-TARGET_DENS)/TARGET_DENS)
    rows.append(dict(sigma=str(sg),foot=str(ft),thr=th,recall=round(rec,4),
                     dens=round(den,0), est_jaccard=round(rec*rec*pen,4),
                     min_ds_recall=round(min(np.mean(v) for v in per_ds.values()),3)))
    print(f"{str(sg):12s} {str(ft):12s} {th:9s} recall={rec:.3f} dens={den:5.0f} est_J={rec*rec*pen:.3f}")
res=pd.DataFrame(rows).sort_values("est_jaccard",ascending=False)
print(f"\n{len(rows)} konfig, {time.time()-t0:.0f}s")
print("\n=== EN IYI 10 ===")
print(res.head(10).to_string(index=False))

### 3a · Görselleştirme — recall vs yoğunluk

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(14,5))
sc=ax[0].scatter(res.dens,res.recall,c=res.est_jaccard,cmap="viridis",s=60)
ax[0].axvline(TARGET_DENS,color="r",ls="--",label=f"hedef {TARGET_DENS}/kare")
ax[0].set_xlabel("yoğunluk (node/kare)"); ax[0].set_ylabel("recall")
ax[0].set_title("recall vs yoğunluk (renk = tahmini jaccard)"); ax[0].legend()
plt.colorbar(sc,ax=ax[0],label="est_jaccard")
best=res.iloc[0]
ax[1].barh(range(10), res.head(10).est_jaccard[::-1])
ax[1].set_yticks(range(10))
ax[1].set_yticklabels([f"{r.sigma}|{r.foot}|{r.thr}" for r in res.head(10).itertuples()][::-1],fontsize=7)
ax[1].set_xlabel("tahmini jaccard"); ax[1].set_title("en iyi 10 konfig")
plt.tight_layout(); plt.savefig("/kaggle/working/D11_detection_sweep.png",dpi=120,bbox_inches="tight"); plt.show()
print("EN IYI:", dict(best))

## 4 · Anomali: `44b6_0b24845f` neden recall 0.33?
Aynı parametrelerle bir dataset 1.00, diğeri 0.33 veriyor. GT hücreleri sönük mü,
yoksa tespit onları mı kaçırıyor? En iyi konfigle mesafe/parlaklık dağılımına bakalım.

In [ ]:
bs=eval(best.sigma); bf=eval(best.foot); bt=best.thr
print("kullanilan konfig:",bs,bf,bt,"\n")
for nm,t,v,gp in FRAMES:
    c=detect(v,bs,bf,bt)
    if len(c)==0: print(nm,t,"tespit YOK"); continue
    D=cdist(gp*S,c*S); dmin=D.min(1)
    gi=[float(v[int(round(p[0])),int(round(p[1])),int(round(p[2]))]) for p in gp]
    print(f"{nm} t={t:3d} | GT={len(gp):4d} tespit={len(c):4d} "
          f"| recall={float((dmin<=MATCH_UM).mean()):.2f} "
          f"| GT->en yakin tespit medyan={np.median(dmin):5.2f}um "
          f"| GT parlaklik medyan={np.median(gi):6.0f} | hacim p99={np.percentile(v,99):6.0f}")

## 5 · Sonuç → `02_baseline`'a taşı

En iyi konfigürasyonu `02_baseline.ipynb`'deki `SIGMA` / `FOOT` / eşik ayarına yaz ve
tam koşuyu tekrarla.

- [ ] En iyi `sigma`/`foot`/`thr` = **…**
- [ ] Beklenen recall = **…** → tahmini jaccard = **…** (baseline: 0.78 / 0.68)
- [ ] Yoğunluk ~213/kare'ye yaklaştı mı (fazla-tahmin cezası)?
- [ ] Anomali dataset düzeldi mi?